# Fine-Tuning Laya Multilingual en Google Colab Pro GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![Model Checkpoint](https://img.shields.io/badge/%F0%9F%A4%97%20Base%20Model-convaiinnovations%2Flaya--multilingual-blue)](https://huggingface.co/convaiinnovations/laya)

Este notebook orquesta el entrenamiento supervisado (fine-tuning) del modelo `convaiinnovations/laya-multilingual` adaptado a tickets de soporte técnico en español.

### Protocolo de Calidad y Aislamiento Científico:
1. **Entrenamiento:** Se entrena exclusivamente con `data/splits/train.jsonl` (1.021 tickets).
2. **Validación:** Se valida únicamente con `data/splits/val.jsonl` (219 tickets).
3. **Aislamiento Total:** El conjunto de prueba congelado (`data/splits/test.jsonl`) **nunca** se consulta ni se utiliza durante el entrenamiento.
4. **Early Stopping & Checkpoints Duales:** Monitoreo por época de Loss y Macro-F1, guardado del mejor modelo (`best_model.safetensors`), último modelo (`last_model.safetensors`) y detención temprana ante estancamiento.

## 1. Verificación del Entorno GPU y Optimización de VRAM

In [ ]:
!nvidia-smi
import os, torch

assert torch.cuda.is_available(), "¡GPU no detectada! Cambie el entorno a GPU (Entorno de ejecución -> Cambiar tipo -> GPU T4 / V100 / A100)"
props = torch.cuda.get_device_properties(0)
print(f"Conectado a GPU: {props.name} | VRAM Total: {props.total_memory / (1024**3):.2f} GB")

# Optimización de memoria PyTorch para evitar fragmentación de VRAM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 2. Instalación de Dependencias del Proyecto

In [ ]:
%pip install -q -U -r requirements/colab.txt
import laya, transformers, torch
print(f"Laya Version:         {laya.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"PyTorch Version:      {torch.__version__}")

## 3. Verificación de Integridad de Datos y Aislamiento Estricto (Zero-Leakage)

In [ ]:
from pathlib import Path
from src.preprocessing.split import calculate_file_sha256, verify_zero_leakage
from src.training.train import load_jsonl_records

train_path = Path("data/splits/train.jsonl")
val_path = Path("data/splits/val.jsonl")

print(f"Train SHA-256: {calculate_file_sha256(train_path)}")
print(f"Val SHA-256:   {calculate_file_sha256(val_path)}")

train_recs = load_jsonl_records(train_path)
val_recs = load_jsonl_records(val_path)

# Confirmación matemática de cero solapamiento
assert verify_zero_leakage(train_recs, val_recs, [], hash_key="content_hash") is True
print(f"✓ Aislamiento verificado: {len(train_recs)} train, {len(val_recs)} val. Cero colisiones de contenido.")

## 4. Ejecución del Entrenamiento con Early Stopping

In [ ]:
import time
from src.training.train import train_laya

start_t = time.time()
report = train_laya(
    config_path=Path("configs/training.yaml"),
    device_name="cuda" if torch.cuda.is_available() else "cpu",
)
elapsed_min = (time.time() - start_t) / 60.0
print(f"\nTiempo total de entrenamiento: {elapsed_min:.2f} minutos")

## 5. Resumen de Checkpoints Generados y Reporte

In [ ]:
import json
report_path = Path("checkpoints/laya_service_desk/training_report.json")
if report_path.exists():
    data = json.loads(report_path.read_text(encoding="utf-8"))
    print("--- RESUMEN DEL ENTRENAMIENTO ---")
    print(f"Épocas planeadas: {data['hyperparameters']['epochs_planned']}")
    print(f"Épocas ejecutadas: {data['hyperparameters']['epochs_executed']}")
    print(f"Early Stopping activado: {data['early_stopping']['stopped_early']}")
    print(f"Mejor época: {data['early_stopping']['best_epoch']} con {data['early_stopping']['best_metric_name']}={data['early_stopping']['best_metric_value']}")
    print(f"Mejor checkpoint SHA-256: {data['checkpoints']['best']['sha256']}")
    print(f"Último checkpoint SHA-256: {data['checkpoints']['last']['sha256']}")

## 6. Visualización y Publicación de Curvas de Aprendizaje (Pérdida y Métricas)

Los resultados de cada época se persisten de manera estructurada en `training_curves.csv` y `training_curves.json`.
Asimismo, se generan automáticamente gráficos de calidad de publicación a 300 DPI en `reports/figures/`.

In [ ]:
import csv
from pathlib import Path
from IPython.display import Image, display

curves_csv = Path("checkpoints/laya_service_desk/training_curves.csv")
if curves_csv.exists():
    print("--- RESULTADOS TABULARES POR ÉPOCA (training_curves.csv) ---")
    with curves_csv.open("r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            print(" | ".join(f"{cell:<12}" for cell in row))

# Mostrar figuras generadas a 300 DPI para publicación e informe
dashboard_img = Path("reports/figures/training_dashboard.png")
if dashboard_img.exists():
    print("\n--- DASHBOARD INTEGRAL DE RENDIMIENTO ---")
    display(Image(filename=str(dashboard_img)))

loss_img = Path("reports/figures/training_loss_curve.png")
metrics_img = Path("reports/figures/training_metrics_curve.png")
if loss_img.exists() and metrics_img.exists():
    print("\n--- CURVA DE PÉRDIDA ---")
    display(Image(filename=str(loss_img)))
    print("\n--- CURVAS DE MÉTRICAS MULTITAREA ---")
    display(Image(filename=str(metrics_img)))

## 7. Verificación de Inferencia con el Modelo Ajustado

In [ ]:
import laya
from src.preprocessing.format_laya import build_default_laya_questions

# Cargar directamente el modelo fine-tuned exportado
agent = laya.Agent("checkpoints/laya_service_desk", device="cuda" if torch.cuda.is_available() else "cpu")

sample_state = {
    "subject": "Error al conectar a la VPN corporativa",
    "body": "Desde esta mañana la conexión VPN rechaza mis credenciales en la red de la oficina."
}
questions = build_default_laya_questions()

res = agent.predict(sample_state, questions)
print("\n--- Predicciones del Modelo Fine-Tuned ---")
for qid, ans in res.get("answers", {}).items():
    print(f"Tarea '{qid}': {ans.get('choice')} (confianza: {ans.get('confidence', 0):.2f})")
print("Uso de tokens:", res.get("usage"))

## 8. (Opcional) Persistencia de Pesos en Google Drive

In [ ]:
# Descomentar para guardar en Google Drive tras autorización:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/laya_service_desk_checkpoints/
# !cp -r checkpoints/laya_service_desk/ /content/drive/MyDrive/laya_service_desk_checkpoints/
# print("Checkpoints exportados exitosamente a Google Drive.")